# Random Forest - Analise Detalhada

Notebook dedicado a analise aprofundada do modelo Random Forest para predicao do Ponto de Virada (PV) dos alunos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
)
import warnings

from models.preprocessing import obter_dados_e_preprocessar
from models.model_io import save_model, load_model

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

## 1. Carregamento e preprocessamento dos dados

In [ ]:
scaler = StandardScaler()
df, cluster_metricas, metricas_normalizadas = obter_dados_e_preprocessar(
    "dataset/PEDE_PASSOS_DATASET_FIAP.csv", scaler
)

df["Alvo_PV"] = df["Atingiu PV"].apply(
    lambda x: 1 if str(x).strip().upper() == "SIM" else 0
)

X = df[cluster_metricas]
y = df["Alvo_PV"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Amostras de treino: {len(X_train)}")
print(f"Amostras de teste: {len(X_test)}")
print(f"Distribuicao alvo (treino): {y_train.value_counts().to_dict()}")
print(f"Distribuicao alvo (teste): {y_test.value_counts().to_dict()}")

## 2. Treinamento do modelo base

In [ ]:
modelo_rf = RandomForestClassifier(
    n_estimators=150,
    min_samples_leaf=4,
    max_features="sqrt",
    random_state=42,
)
modelo_rf.fit(X_train, y_train)

y_pred = modelo_rf.predict(X_test)
y_proba = modelo_rf.predict_proba(X_test)[:, 1]

print("Relatorio de Classificacao:")
print(classification_report(y_test, y_pred, target_names=["Nao atingiu PV", "Atingiu PV"]))

## 3. Matriz de confusao

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Nao atingiu PV", "Atingiu PV"],
    yticklabels=["Nao atingiu PV", "Atingiu PV"],
    ax=ax,
)
ax.set_xlabel("Previsto", fontsize=12)
ax.set_ylabel("Real", fontsize=12)
ax.set_title("Matriz de Confusao - Random Forest", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Curva ROC e AUC

In [ ]:
fpr, tpr, thresholds_roc = roc_curve(y_test, y_proba)
auc_score = roc_auc_score(y_test, y_proba)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

ax1.plot(fpr, tpr, color="steelblue", linewidth=2, label=f"ROC (AUC = {auc_score:.4f})")
ax1.plot([0, 1], [0, 1], color="gray", linestyle="--", linewidth=1)
ax1.set_xlabel("Taxa de Falsos Positivos", fontsize=12)
ax1.set_ylabel("Taxa de Verdadeiros Positivos", fontsize=12)
ax1.set_title("Curva ROC", fontsize=14, fontweight="bold")
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

precision, recall, thresholds_pr = precision_recall_curve(y_test, y_proba)
ap_score = average_precision_score(y_test, y_proba)

ax2.plot(recall, precision, color="darkorange", linewidth=2, label=f"PR (AP = {ap_score:.4f})")
ax2.set_xlabel("Recall", fontsize=12)
ax2.set_ylabel("Precision", fontsize=12)
ax2.set_title("Curva Precision-Recall", fontsize=14, fontweight="bold")
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"AUC-ROC: {auc_score:.4f}")
print(f"Average Precision: {ap_score:.4f}")

## 5. Importancia das features

In [ ]:
importancias = pd.Series(modelo_rf.feature_importances_, index=cluster_metricas).sort_values(
    ascending=True
)

fig, ax = plt.subplots(figsize=(10, 6))
importancias.plot(kind="barh", ax=ax, color="steelblue", edgecolor="black")
ax.set_xlabel("Importancia", fontsize=12)
ax.set_title("Importancia das Features - Random Forest", fontsize=14, fontweight="bold")
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

print("\nImportancia das features (ordem decrescente):")
for feat, imp in importancias.sort_values(ascending=False).items():
    print(f"  {feat}: {imp:.4f}")

## 6. Validacao cruzada

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores_accuracy = cross_val_score(modelo_rf, X, y, cv=cv, scoring="accuracy")
scores_f1 = cross_val_score(modelo_rf, X, y, cv=cv, scoring="f1")
scores_roc = cross_val_score(modelo_rf, X, y, cv=cv, scoring="roc_auc")

resultados_cv = pd.DataFrame({
    "Fold": range(1, 6),
    "Accuracy": scores_accuracy,
    "F1-Score": scores_f1,
    "AUC-ROC": scores_roc,
})

print("Resultados da Validacao Cruzada (5-Fold):")
print(resultados_cv.to_string(index=False))
print(f"\nAccuracy: {scores_accuracy.mean():.4f} (+/- {scores_accuracy.std():.4f})")
print(f"F1-Score: {scores_f1.mean():.4f} (+/- {scores_f1.std():.4f})")
print(f"AUC-ROC:  {scores_roc.mean():.4f} (+/- {scores_roc.std():.4f})")

fig, ax = plt.subplots(figsize=(10, 5))
x_pos = np.arange(5)
width = 0.25
ax.bar(x_pos - width, scores_accuracy, width, label="Accuracy", color="#3498db")
ax.bar(x_pos, scores_f1, width, label="F1-Score", color="#2ecc71")
ax.bar(x_pos + width, scores_roc, width, label="AUC-ROC", color="#e74c3c")
ax.set_xticks(x_pos)
ax.set_xticklabels([f"Fold {i}" for i in range(1, 6)])
ax.set_ylabel("Score", fontsize=12)
ax.set_title("Validacao Cruzada por Fold", fontsize=14, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## 7. Comparacao de hiperparametros

In [ ]:
configs = [
    {"n_estimators": 50, "min_samples_leaf": 1, "max_features": "sqrt"},
    {"n_estimators": 100, "min_samples_leaf": 2, "max_features": "sqrt"},
    {"n_estimators": 150, "min_samples_leaf": 4, "max_features": "sqrt"},
    {"n_estimators": 200, "min_samples_leaf": 4, "max_features": "log2"},
    {"n_estimators": 300, "min_samples_leaf": 6, "max_features": "sqrt"},
]

resultados = []
for cfg in configs:
    rf_temp = RandomForestClassifier(random_state=42, **cfg)
    scores = cross_val_score(rf_temp, X, y, cv=cv, scoring="roc_auc")
    resultados.append({
        "n_estimators": cfg["n_estimators"],
        "min_samples_leaf": cfg["min_samples_leaf"],
        "max_features": cfg["max_features"],
        "AUC_mean": scores.mean(),
        "AUC_std": scores.std(),
    })

df_resultados = pd.DataFrame(resultados).sort_values("AUC_mean", ascending=False)
print("Comparacao de hiperparametros (5-Fold CV AUC-ROC):")
print(df_resultados.to_string(index=False))

## 8. Distribuicao de probabilidades por classe

In [ ]:
df["Probabilidade_PV"] = modelo_rf.predict_proba(X)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(
    df[df["Alvo_PV"] == 0]["Probabilidade_PV"],
    bins=30, alpha=0.7, color="#e74c3c", edgecolor="black",
    label="Nao atingiu PV",
)
axes[0].hist(
    df[df["Alvo_PV"] == 1]["Probabilidade_PV"],
    bins=30, alpha=0.7, color="#2ecc71", edgecolor="black",
    label="Atingiu PV",
)
axes[0].set_xlabel("Probabilidade PV", fontsize=12)
axes[0].set_ylabel("Frequencia", fontsize=12)
axes[0].set_title("Distribuicao de Probabilidades por Classe", fontsize=13, fontweight="bold")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].boxplot(
    [df[df["Alvo_PV"] == 0]["Probabilidade_PV"], df[df["Alvo_PV"] == 1]["Probabilidade_PV"]],
    labels=["Nao atingiu PV", "Atingiu PV"],
)
axes[1].set_ylabel("Probabilidade PV", fontsize=12)
axes[1].set_title("Boxplot de Probabilidades por Classe", fontsize=13, fontweight="bold")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Exportar modelo treinado

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=4, random_state=42, n_init=20)
df["Grupo_ID"] = kmeans.fit_predict(metricas_normalizadas)

save_model(scaler, kmeans, modelo_rf, cluster_metricas, "artifacts")
print("Artefatos do modelo exportados para artifacts/")